In [1]:
import os
for f in sorted(os.listdir("/kaggle/input/datasets/shadman1028/cicids2017-official-flow-feature-csv-files")):
    print(f)

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Wednesday-workingHours.pcap_ISCX.csv


In [2]:
#Inspect one file
import pandas as pd
import numpy as np

DATA_DIR = "/kaggle/input/datasets/shadman1028/cicids2017-official-flow-feature-csv-files"

df = pd.read_csv(f"{DATA_DIR}/Monday-WorkingHours.pcap_ISCX.csv", low_memory=False)
df.columns = df.columns.str.strip()

print("Shape:", df.shape)
print("\nColumns:")
for col in df.columns:
    print(" ", col)


Shape: (529918, 79)

Columns:
  Destination Port
  Flow Duration
  Total Fwd Packets
  Total Backward Packets
  Total Length of Fwd Packets
  Total Length of Bwd Packets
  Fwd Packet Length Max
  Fwd Packet Length Min
  Fwd Packet Length Mean
  Fwd Packet Length Std
  Bwd Packet Length Max
  Bwd Packet Length Min
  Bwd Packet Length Mean
  Bwd Packet Length Std
  Flow Bytes/s
  Flow Packets/s
  Flow IAT Mean
  Flow IAT Std
  Flow IAT Max
  Flow IAT Min
  Fwd IAT Total
  Fwd IAT Mean
  Fwd IAT Std
  Fwd IAT Max
  Fwd IAT Min
  Bwd IAT Total
  Bwd IAT Mean
  Bwd IAT Std
  Bwd IAT Max
  Bwd IAT Min
  Fwd PSH Flags
  Bwd PSH Flags
  Fwd URG Flags
  Bwd URG Flags
  Fwd Header Length
  Bwd Header Length
  Fwd Packets/s
  Bwd Packets/s
  Min Packet Length
  Max Packet Length
  Packet Length Mean
  Packet Length Std
  Packet Length Variance
  FIN Flag Count
  SYN Flag Count
  RST Flag Count
  PSH Flag Count
  ACK Flag Count
  URG Flag Count
  CWE Flag Count
  ECE Flag Count
  Down/Up Ratio
  A

In [4]:
#Check labels and data quality
print("=== Label Distribution ===")
print(df['Label'].value_counts())

print("\n=== Inf values per column ===")
inf_counts = (df == np.inf).sum() + (df == -np.inf).sum()
print(inf_counts[inf_counts > 0])

print("\n=== NaN values per column ===")
nan_counts = df.isnull().sum()
print(nan_counts[nan_counts > 0])

=== Label Distribution ===
Label
BENIGN    529918
Name: count, dtype: int64

=== Inf values per column ===
Flow Bytes/s      373
Flow Packets/s    437
dtype: int64

=== NaN values per column ===
Flow Bytes/s    64
dtype: int64


In [5]:
#Load all files and check full label distribution
import os

dfs = []
for f in sorted(os.listdir(DATA_DIR)):
    path = os.path.join(DATA_DIR, f)
    tmp = pd.read_csv(path, low_memory=False)
    tmp.columns = tmp.columns.str.strip()
    dfs.append(tmp)
    print(f"{f}: {len(tmp)} rows")

df_all = pd.concat(dfs, ignore_index=True)
print(f"\nTotal rows: {len(df_all)}")
print("\n=== Full Label Distribution ===")
print(df_all['Label'].value_counts())

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 225745 rows
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 286467 rows
Friday-WorkingHours-Morning.pcap_ISCX.csv: 191033 rows
Monday-WorkingHours.pcap_ISCX.csv: 529918 rows
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 288602 rows
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 170366 rows
Tuesday-WorkingHours.pcap_ISCX.csv: 445909 rows
Wednesday-workingHours.pcap_ISCX.csv: 692703 rows

Total rows: 2830743

=== Full Label Distribution ===
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration   

In [6]:
#Check inf/NaN across all files
print("=== Inf values (all data) ===")
inf_counts = (df_all == np.inf).sum() + (df_all == -np.inf).sum()
print(inf_counts[inf_counts > 0])

print("\n=== NaN values (all data) ===")
nan_counts = df_all.isnull().sum()
print(nan_counts[nan_counts > 0])

print("\n=== Classes with < 100 samples (will drop) ===")
vc = df_all['Label'].value_counts()
print(vc[vc < 100])


=== Inf values (all data) ===
Flow Bytes/s      1509
Flow Packets/s    2867
dtype: int64

=== NaN values (all data) ===
Flow Bytes/s    1358
dtype: int64

=== Classes with < 100 samples (will drop) ===
Label
Infiltration                  36
Web Attack � Sql Injection    21
Heartbleed                    11
Name: count, dtype: int64


In [7]:
#Find constant and duplicate columns
# Constant columns (only 1 unique value — useless for a model)
numeric_cols = df_all.drop(columns=['Label']).select_dtypes(include=np.number).columns
constant_cols = [c for c in numeric_cols if df_all[c].nunique() <= 1]
print("=== Constant columns ===")
print(constant_cols)

# Near-constant (>99.9% same value)
near_constant = []
for c in numeric_cols:
    top_freq = df_all[c].value_counts(normalize=True).iloc[0]
    if top_freq > 0.999:
        near_constant.append((c, round(top_freq * 100, 2)))
print("\n=== Near-constant columns (>99.9% one value) ===")
for c, pct in near_constant:
    print(f"  {c}: {pct}%")

# Confirm the known duplicate
print("\n=== Fwd Header Length vs Fwd Header Length.1 identical? ===")
print((df_all['Fwd Header Length'] == df_all['Fwd Header Length.1']).all())

=== Constant columns ===
['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']

=== Near-constant columns (>99.9% one value) ===
  Bwd PSH Flags: 100.0%
  Fwd URG Flags: 99.99%
  Bwd URG Flags: 100.0%
  RST Flag Count: 99.98%
  CWE Flag Count: 99.99%
  ECE Flag Count: 99.98%
  Fwd Avg Bytes/Bulk: 100.0%
  Fwd Avg Packets/Bulk: 100.0%
  Fwd Avg Bulk Rate: 100.0%
  Bwd Avg Bytes/Bulk: 100.0%
  Bwd Avg Packets/Bulk: 100.0%
  Bwd Avg Bulk Rate: 100.0%

=== Fwd Header Length vs Fwd Header Length.1 identical? ===
True


In [8]:
#Define FEATURE_COLUMNS
DROP_COLS = [
    'Label',
    'Fwd Header Length.1',       # exact duplicate
    # constant
    'Bwd PSH Flags', 'Bwd URG Flags',
    'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate',
    'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate',
    # near-constant
    'Fwd URG Flags', 'RST Flag Count', 'CWE Flag Count', 'ECE Flag Count',
]

FEATURE_COLUMNS = [c for c in df_all.columns if c not in DROP_COLS]

print(f"Feature count: {len(FEATURE_COLUMNS)}")
print("\nFEATURE_COLUMNS =", FEATURE_COLUMNS)


Feature count: 65

FEATURE_COLUMNS = ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'Down/Up Ratio', 'Average Packet Size', 'Avg Fwd Segment Size', '

In [10]:
#Clean, encode, split, SMOTE, scale, save
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import joblib

# 1. Drop rare classes
rare = ['Infiltration', 'Web Attack \ufffd Sql Injection', 'Heartbleed']
df_clean = df_all[~df_all['Label'].isin(rare)].copy()

# 2. Fix encoding in label names
df_clean['Label'] = df_clean['Label'].str.replace('Web Attack \ufffd', 'Web Attack -', regex=False)

# 3. Replace inf with NaN then drop
df_clean = df_clean.replace([np.inf, -np.inf], np.nan)
df_clean = df_clean.dropna()

print(f"Rows after cleaning: {len(df_clean)}")
print(f"Labels remaining:\n{df_clean['Label'].value_counts()}")

Rows after cleaning: 2827808
Labels remaining:
Label
BENIGN                      2271320
DoS Hulk                     230124
PortScan                     158804
DDoS                         128025
DoS GoldenEye                 10293
FTP-Patator                    7935
SSH-Patator                    5897
DoS slowloris                  5796
DoS Slowhttptest               5499
Bot                            1956
Web Attack - Brute Force       1507
Web Attack - XSS                652
Name: count, dtype: int64


In [11]:
#Smarter SMOTE with cap
import os
from collections import Counter
os.makedirs("/kaggle/working/processed", exist_ok=True)

le = LabelEncoder()
y = le.fit_transform(df_clean['Label'].values)
X = df_clean[FEATURE_COLUMNS].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Step 1: manually undersample BENIGN to 200k
benign_class = le.transform(['BENIGN'])[0]
benign_idx = np.where(y_train == benign_class)[0]
other_idx  = np.where(y_train != benign_class)[0]
rng = np.random.default_rng(42)
benign_keep = rng.choice(benign_idx, size=200_000, replace=False)
keep = np.concatenate([benign_keep, other_idx])
X_train, y_train = X_train[keep], y_train[keep]

print("After undersampling BENIGN:")
for cls, cnt in sorted(Counter(y_train).items()):
    print(f"  {le.classes_[cls]}: {cnt}")

# Step 2: SMOTE only classes below 50k
counts = Counter(y_train)
sampling_strategy = {
    cls: 50_000 for cls, cnt in counts.items() if cnt < 50_000
}
print("\nSMOTE will boost:", {le.classes_[k]: v for k, v in sampling_strategy.items()})

print("\nApplying SMOTE...")
sm = SMOTE(random_state=42, sampling_strategy=sampling_strategy)
X_train, y_train = sm.fit_resample(X_train, y_train)
print(f"After SMOTE — Train: {X_train.shape}")

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Save
np.save("/kaggle/working/processed/X_train.npy", X_train)
np.save("/kaggle/working/processed/X_test.npy",  X_test)
np.save("/kaggle/working/processed/y_train.npy", y_train)
np.save("/kaggle/working/processed/y_test.npy",  y_test)
joblib.dump(scaler, "/kaggle/working/processed/scaler.pkl")
joblib.dump(le,     "/kaggle/working/processed/label_encoder.pkl")

print("\nDone.")

After undersampling BENIGN:
  BENIGN: 200000
  Bot: 1565
  DDoS: 102420
  DoS GoldenEye: 8234
  DoS Hulk: 184099
  DoS Slowhttptest: 4399
  DoS slowloris: 4637
  FTP-Patator: 6348
  PortScan: 127043
  SSH-Patator: 4717
  Web Attack - Brute Force: 1206
  Web Attack - XSS: 522

SMOTE will boost: {'SSH-Patator': 50000, 'DoS GoldenEye': 50000, 'DoS slowloris': 50000, 'FTP-Patator': 50000, 'DoS Slowhttptest': 50000, 'Bot': 50000, 'Web Attack - XSS': 50000, 'Web Attack - Brute Force': 50000}

Applying SMOTE...
After SMOTE — Train: (1013562, 65)

Done.


In [12]:
#Train Random Forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
import joblib

os.makedirs("/kaggle/working/models", exist_ok=True)

X_train = np.load("/kaggle/working/processed/X_train.npy")
X_test  = np.load("/kaggle/working/processed/X_test.npy")
y_train = np.load("/kaggle/working/processed/y_train.npy")
y_test  = np.load("/kaggle/working/processed/y_test.npy")

print("Training Random Forest...")
rf = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

preds = rf.predict(X_test)
rf_f1 = f1_score(y_test, preds, average="weighted")

print(f"\nWeighted F1: {rf_f1:.4f}")
print("\n", classification_report(y_test, preds, target_names=le.classes_))

joblib.dump(rf, "/kaggle/working/models/rf_model.pkl")
print("Saved to /kaggle/working/models/rf_model.pkl")

Training Random Forest...

Weighted F1: 0.9981

                           precision    recall  f1-score   support

                  BENIGN       1.00      1.00      1.00    454264
                     Bot       0.46      0.99      0.63       391
                    DDoS       1.00      1.00      1.00     25605
           DoS GoldenEye       0.99      1.00      0.99      2059
                DoS Hulk       1.00      1.00      1.00     46025
        DoS Slowhttptest       0.99      0.99      0.99      1100
           DoS slowloris       0.99      1.00      1.00      1159
             FTP-Patator       1.00      1.00      1.00      1587
                PortScan       0.99      1.00      1.00     31761
             SSH-Patator       1.00      1.00      1.00      1180
Web Attack - Brute Force       0.79      0.58      0.67       301
        Web Attack - XSS       0.36      0.68      0.48       130

                accuracy                           1.00    565562
               macro avg 

In [13]:
#Train LSTM
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

SEQ_LEN = 5
FPS     = 65 // SEQ_LEN   # 13 features per step, no trimming needed
HIDDEN  = 128
LAYERS  = 2
EPOCHS  = 10
BATCH   = 512
LR      = 1e-3
NUM_CLASSES = len(le.classes_)

X_tr = torch.tensor(X_train.reshape(-1, SEQ_LEN, FPS), dtype=torch.float32)
X_te = torch.tensor(X_test.reshape(-1, SEQ_LEN, FPS),  dtype=torch.float32)
y_tr = torch.tensor(y_train, dtype=torch.long)
y_te = torch.tensor(y_test,  dtype=torch.long)

loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = LSTMClassifier(FPS, HIDDEN, LAYERS, NUM_CLASSES).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(loader):.4f}")

model.eval()
with torch.no_grad():
    lstm_preds = model(X_te.to(device)).argmax(dim=1).cpu().numpy()

lstm_f1 = f1_score(y_test, lstm_preds, average="weighted")
print(f"\nWeighted F1: {lstm_f1:.4f}")
print("\n", classification_report(y_test, lstm_preds, target_names=le.classes_))

torch.save(model.state_dict(), "/kaggle/working/models/lstm_model.pt")
print("Saved to /kaggle/working/models/lstm_model.pt")

Using device: cuda
Epoch 1/10 | Loss: 0.2727
Epoch 2/10 | Loss: 0.1373
Epoch 3/10 | Loss: 0.1282
Epoch 4/10 | Loss: 0.1225
Epoch 5/10 | Loss: 0.1197
Epoch 6/10 | Loss: 0.1138
Epoch 7/10 | Loss: 0.1122
Epoch 8/10 | Loss: 0.1096
Epoch 9/10 | Loss: 0.1052
Epoch 10/10 | Loss: 0.1048


OutOfMemoryError: CUDA out of memory. Tried to allocate 32.79 GiB. GPU 0 has a total capacity of 14.56 GiB of which 10.57 GiB is free. Including non-PyTorch memory, this process has 3.99 GiB memory in use. Of the allocated memory 3.80 GiB is allocated by PyTorch, and 50.11 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [15]:
#Evaluate + save
model.eval()
all_preds = []

with torch.no_grad():
    for i in range(0, len(X_te), 4096):
        batch = X_te[i:i+4096].to(device)
        preds = model(batch).argmax(dim=1).cpu().numpy()
        all_preds.append(preds)

lstm_preds = np.concatenate(all_preds)
lstm_f1 = f1_score(y_test, lstm_preds, average="weighted")
print(f"Weighted F1: {lstm_f1:.4f}")
print("\n", classification_report(y_test, lstm_preds, target_names=le.classes_))

# Save now
torch.save(model.state_dict(), "/kaggle/working/models/lstm_model.pt")
print("\nSaved to /kaggle/working/models/lstm_model.pt")

Weighted F1: 0.9746

                           precision    recall  f1-score   support

                  BENIGN       1.00      0.96      0.98    454264
                     Bot       0.09      0.99      0.17       391
                    DDoS       0.95      1.00      0.97     25605
           DoS GoldenEye       0.93      1.00      0.96      2059
                DoS Hulk       0.95      1.00      0.97     46025
        DoS Slowhttptest       0.87      0.99      0.92      1100
           DoS slowloris       0.94      0.99      0.97      1159
             FTP-Patator       0.98      1.00      0.99      1587
                PortScan       0.83      1.00      0.91     31761
             SSH-Patator       0.83      0.99      0.90      1180
Web Attack - Brute Force       0.15      0.50      0.23       301
        Web Attack - XSS       0.08      0.68      0.14       130

                accuracy                           0.97    565562
               macro avg       0.72      0.93      0